# Single-Layer RNN

This tutorial follows one recurrent model through the complete compiler workflow. We compile a single-layer RNN, inspect the human-readable compilation report, and then examine the `ETraceGraph` relations that drive online learning.

## Define and Compile the Model

The model contains one `ValinaRNNCell` and one linear readout. The recurrent cell owns the temporal hidden state, while the readout maps that state to the final output.

In [1]:
import jax
import jax.numpy as jnp
import brainstate
import braintrace

In [2]:
class SingleLayerRNN(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn = braintrace.nn.ValinaRNNCell(n_in, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        return self.out(self.rnn(x))


model = SingleLayerRNN(10, 32, 5)

# braintrace.compile initialises states, compiles the ETP graph, and returns a ready learner.
# We compile for a single unbatched sample (no batch_size), so the hidden state is (32,) and
# the recurrent op is the matrix-vector primitive etp_mv. verbose=2 prints full diagnostics.
learner = braintrace.compile(model, braintrace.D_RTRL, jnp.zeros(10), verbose=2)
learner.show_graph()

The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)


Compiler diagnostics (warnings / errors):

   [warning] relation_excluded_non_temporal: ETP primitive etp_mv (weight=('out', 'weight')) has no connected hidden states. It will be treated as a non-temporal parameter.



The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)





## Read the Compiler Output

The compiler identifies one hidden group at `('rnn', 'h')`. The recurrent weight at `('rnn', 'W', 'weight')` reaches that hidden state through an ETP primitive, so the learner maintains an eligibility trace for it.

The readout weight at `('out', 'weight')` is different. Its output does not feed a recurrent hidden state, so it is reported as `relation_excluded_non_temporal`. It remains trainable through the loss, but it does not require a temporal eligibility trace.

## Using `learner.report` -- the `CompilationReport`

Every learner returned by `braintrace.compile` exposes a `CompilationReport` through `learner.report`. The report is the concise view of what compilation included, excluded, or diagnosed:

- `report.counts` summarizes hidden groups, ETP weights, excluded weights, warnings, and errors.
- `report.etrace_weights` lists parameter paths that participate in eligibility tracing.
- `report.excluded_weights` pairs excluded parameter paths with their reasons.
- `report.dynamic_states` records non-hidden dynamic states discovered during tracing.
- `report.diagnostics` contains the complete `CompilationRecord` sequence.

In [3]:
# report.show(level) prints a structured summary at the requested verbosity.
# level=1 shows hidden groups, etrace weights, and excluded weights.
learner.report.show(1)

# Programmatic access to the summary counts
print("Counts:", learner.report.counts)

# Which weights participate in online learning?
print("ETrace weights:", learner.report.etrace_weights)

# Which weights were excluded (e.g., non-temporal readouts)?
print("Excluded weights:", learner.report.excluded_weights)

The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)



Counts: {'hidden_groups': 1, 'etrace_weights': 2, 'excluded_weights': 1, 'warnings': 1, 'errors': 0}
ETrace weights: [(('rnn', 'W', 'weight'), [0]), (('rnn', 'W', 'weight'), [0])]
Excluded weights: [(('out', 'weight'), 'relation_excluded_non_temporal')]


## Understanding `ETraceGraph`

The report summarizes decisions; `learner.graph` exposes their structural representation. Its central fields are:

| Field | Meaning |
|---|---|
| `module_info` | Traced Jaxpr and model-state mappings |
| `hidden_groups` | Hidden states updated as recurrent groups |
| `hid_path_to_group` | Hidden-state path to group mapping |
| `hidden_param_op_relations` | Parameter, ETP primitive, and hidden-group relations |
| `hidden_perturb` | Perturbation structure used for hidden Jacobians |
| `diagnostics` | Included and excluded compiler decisions |

Inspecting the relation itself confirms which trainable path, primitive, and hidden group produced the summary above.

In [4]:
graph = learner.graph

print("=== Hidden Groups ===")
for g in graph.hidden_groups:
    print(f"  Group {g.index}: {g.num_state} state(s), shape {g.varshape}")
    print(f"    Paths: {g.hidden_paths}")

print("\n=== Weight-Primitive-Hidden Relations ===")
for i, r in enumerate(graph.hidden_param_op_relations):
    print(f"  Relation {i}:")
    # ``trainable_paths`` is a dict {trainable key -> owning ParamState path};
    # a single primitive may own several (e.g. {weight, bias}).
    print(f"    Trainable paths: {r.trainable_paths}")
    print(f"    Primitive: {r.primitive}")
    print(f"    Hidden groups: {[g.index for g in r.hidden_groups]}")

print(f"\n=== Perturbation ===")
print(f"  Has perturbation: {graph.hidden_perturb is not None}")

=== Hidden Groups ===
  Group 0: 1 state(s), shape (32,)
    Paths: [('rnn', 'h')]

=== Weight-Primitive-Hidden Relations ===
  Relation 0:
    Trainable paths: {'weight': ('rnn', 'W', 'weight'), 'bias': ('rnn', 'W', 'weight')}
    Primitive: etp_mv
    Hidden groups: [0]

=== Perturbation ===
  Has perturbation: True


## Using `compile_etrace_graph` Directly

`braintrace.compile(...)` is the standard entry point because it initializes states, compiles the graph, and returns a ready learner. For structural debugging or custom algorithm development, `braintrace.compile_etrace_graph(...)` returns the same `ETraceGraph` without constructing an algorithm wrapper.

In [5]:
model_direct = SingleLayerRNN(10, 32, 5)
brainstate.nn.init_all_states(model_direct)

graph_direct = braintrace.compile_etrace_graph(model_direct, jnp.zeros(10))

print(f"Number of hidden groups: {len(graph_direct.hidden_groups)}")
print(f"Number of relations: {len(graph_direct.hidden_param_op_relations)}")
print(f"Has perturbation: {graph_direct.hidden_perturb is not None}")

print("\nGraph fields:")
for key in graph_direct.dict().keys():
    print(f"  {key}")

Number of hidden groups: 1
Number of relations: 1
Has perturbation: True

Graph fields:
  module_info
  hidden_groups
  hid_path_to_group
  hidden_param_op_relations
  hidden_perturb
  diagnostics


## Summary

The single-layer case establishes the full inspection sequence: compile the model, read the report, and confirm the underlying graph relations. These diagnostics establish what the compiler selected; they do not establish gradient correctness. Continue with [Two-Layer RNN](two_layer_rnn.ipynb) to see how the same workflow separates multiple recurrent layers.